In [1]:
from pathlib import Path
if not Path('jw_utils').exists(): 
    !git clone https://github.com/JonWinkelman/jw_utils.git
if not Path('orthofinder_utils').exists(): 
    !git clone https://github.com/JonWinkelman/orthofinder_utils.git
import pandas as pd 
from orthofinder_utils import dash_ortho_parser_d as dop
from jw_utils import parse_gff as pgf
from jw_utils import itol_annotations as ia 
from plotly import express as px
from jw_utils import plotly_preferences as pprefs
import shutil

In [2]:
# dirs to make
results_dir = Path('./results/3_operon_structure_analysis')
newick_tree_dir = Path('./results/3_operon_structure_analysis/newick_trees/')
itol_annots_dir = results_dir /  'itol_annotations'

for d in [results_dir, newick_tree_dir, itol_annots_dir]:
    d.mkdir(exist_ok=True, parents=True)

 | common name |	locus tag | HOG | Notes |
|-------------|-----------|-------------|----------------|
|**AstR**|
|AstR|ACX60_12910|N0.HOG0002984|
|**AstNOP**| 
|AstN|ACX60_12905|N0.HOG0002567|
|AstO|ACX60_12900|N0.HOG0002483|	
|AstP|ACX60_12895|N0.HOG0000027|
|**AstA1**|  
|Ast1|ACX60_01815|N0.HOG0002209|glutamate dehydrogenase|
|Ast2|ACX60_01820|N0.HOG0002567|acetylornithine aminotransferase|
|AstA|ACX60_01825|N0.HOG0002483|arginine N-succinyltransferase|
|Ast4|ACX60_01830|N0.HOG0002860|succinylglutamate-semialdehyde|
|Ast5|ACX60_01835|N0.HOG0002934|N-succinylarginine|
|Ast6|ACX60_01840|N0.HOG0002935|succinylglutamate|

In [4]:
dop_obj_acinetobacter2 = dop.DashOrthoParser('../../all_dash_apps/dash_app_Acinetobacters2/data/')
# dop_obj_acinetobacter = dop.DashOrthoParser('../../all_dash_apps/dash_app_Acinetobacters/data')

In [5]:
a_baum_acc = 'GCA_001077675.1'
gene_to_prot_d =dop_obj_acinetobacter2.gene_to_prot_d(a_baum_acc)

In [6]:
genes_of_interest = {
            "ACX60_12910":'AstR',
            "ACX60_12905":'AstN',
            "ACX60_12900":'AstO',
            "ACX60_12895":'AstP',
}

In [164]:
### make annotations from Acinetobacter2 orthofinder

In [7]:
protein_to_HOG_df = dop_obj_acinetobacter2.protein_to_HOG_df(a_baum_acc)
prot_HOG_d = protein_to_HOG_df['HOG'].to_dict()
gene2HOG_d = {gene:prot_HOG_d[gene_to_prot_d[gene]] for gene in genes_of_interest.keys()}

tree_fp = '../../all_dash_apps/dash_app_Acinetobacters2/data/Species_Tree/bac120_input_tree.rooted.nwk'
local_acineto2_tree_fp = newick_tree_dir / 'Acineto2_bac120_tree.nwk'
shutil.copy(tree_fp, local_acineto2_tree_fp)
itol_annots_dir.mkdir(exist_ok=True, parents=True)


def make_itol_multi_bargraph_annot(dop_obj, out_fp, gene2HOG_d, binary=False):
    count_dict = {acc:[] for acc in dop_obj_acinetobacter2.accessions}
    name_list = []
    hexcolors_all = []
    for gene, HOG in gene2HOG_d.items():
        prot_counts_in_HOG = dop_obj_acinetobacter2.prot_counts_in_HOG(HOG)
        name_list.append(f'{gene}_{HOG}_{genes_of_interest[gene]}')
        for acc, cnt in prot_counts_in_HOG.items():
            if binary:
                if cnt >0:
                    cnt=1
            count_dict[acc].append(cnt)
    hexcolors = [pprefs.rgb_to_hex(c) for c in pprefs.interpolate_colorscale('earth', n=len(gene2HOG_d))]
    ia.make_itol_multi_bargraph_dataset(out_fp, count_dict, name_list, hexcolors=hexcolors)
    print(f'file written to {out_fp}')
    return count_dict, hexcolors, name_list
    
out_fp = itol_annots_dir / f'MULTI_BARGRAPH_cnts_SpeciesTree_rooted_node_labels.txt'
count_dict_full, hexcolors, name_list = make_itol_multi_bargraph_annot(dop_obj_acinetobacter2, out_fp, gene2HOG_d)
out_fp = itol_annots_dir / f'MULTI_BARGRAPH_binary_SpeciesTree_rooted_node_labels_binary.txt'
count_dict_binary, hexcolors, name_list = make_itol_multi_bargraph_annot(dop_obj_acinetobacter2, out_fp, gene2HOG_d, binary=True)

file written to results/3_operon_structure_analysis/itol_annotations/MULTI_BARGRAPH_cnts_SpeciesTree_rooted_node_labels.txt
file written to results/3_operon_structure_analysis/itol_annotations/MULTI_BARGRAPH_binary_SpeciesTree_rooted_node_labels_binary.txt


In [15]:
itol_annots_dir

PosixPath('results/3_operon_structure_analysis/itol_annotations')

In [14]:
df = pd.DataFrame()
df[name_list] = pd.Series(count_dict_full).to_list()
df.index = count_dict_full.keys()
df['Organism_name']= [dop_obj_acinetobacter2.accession_to_name[acc] for acc in df.index]
df = df.reset_index().set_index('Organism_name').reset_index().set_index('index')
df.to_csv(itol_annots_dir / 'MULTI_BARGRAPH_cnts_SpeciesTree_rooted_node_labels.csv')

In [8]:
ia.relabel_itol_treeleafs(dop_obj_acinetobacter2.accession_to_name, itol_annots_dir / 'RELABLE_Acineto2_bac120_tree.rooted.txt')

In [36]:
## get species tree 

126

In [9]:
from Bio import Phylo
AstR_OG = dop_obj_acinetobacter2.HOG_OG_dict()['N0.HOG0002984']
astr_resolved_tree_fp = f'../../all_dash_apps/dash_app_Acinetobacters2/data/Resolved_Gene_Trees/{AstR_OG}_tree.txt'
astr_resolved_tree = Phylo.read(astr_resolved_tree_fp, format='newick')
Phylo.write(astr_resolved_tree, newick_tree_dir / f"{AstR_OG}_tree_resolved.txt", format='newick')

1

In [10]:
astr_resolved_tree_leaves = [cl.name for cl in astr_resolved_tree.get_terminals()]
relable_d = {}
for leafname in astr_resolved_tree_leaves: 
    raw_acc = leafname[:15]
    acc     = raw_acc[::-1].replace('_', '.', 1)[::-1]
    orgname = dop_obj_acinetobacter2.accession_to_name[acc].split(' ')
    orgname = '_'.join([f[:7] for f in orgname])
    newname = f"{leafname[16:]}_{orgname}"
    relable_d[leafname] = newname
out_fp = itol_annots_dir / 'RELABLE_OG0002505_tree.txt'
ia.relabel_itol_treeleafs(relable_d, out_fp)